# Guide to model training and inference

This shows the basics of implementing models for video summarization, while also noting 

In [1]:
from Data import MultiH5Loader
from Models import PGL_SUM
import json
import os
import pytorch_lightning as pl
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import torch.nn.functional as F
from collections.abc import Callable
from Utils import process_and_route_single
from Data import batch_collate_fn

/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Initial Steps

We hold the splits in jsons
The splits have the following structure

Train:

Dataset_name/key_name


Val:

Dataset_name/key_name

Test:

Dataset_name/key_name

In [2]:
split_file = 'Splits/tvsum_can_1.json'
split_name_train= 'train'
split_name_test = 'test'
cross_val_idx = 2
feature_name = 'googlenet'
included_dataset = ['tvsum']


In [3]:
import h5py
base_file_path = 'Data/h5Datasets'

In [4]:
class MultiH5Loader (Dataset):
  def __init__(self,split_file,split_name,cross_val_idx,feature_name,included_dataset):
    with open(split_file,'r') as f:
      self.split_file = json.load(f)
    self.data_points = self.split_file[cross_val_idx][split_name]
    self.feature_name = feature_name
    self._create_data_dict(included_dataset)

  def __len__(self):
    return len(self.data_points)

  def _create_data_dict(self,included_datasets):
    self.dataset_dict = {}
    for dataset in included_datasets:
      data_paths = os.path.join(base_file_path,f'{self.feature_name}',f'{self.feature_name}_{dataset}.h5')
      self.dataset_dict[dataset]= h5py.File(data_paths,'r')


  def __getitem__(self,idx):
    data_point = self.data_points[idx]
    dataset,video_index = data_point.split('/')
    features = self.dataset_dict[dataset][video_index]['features'][...]
    gtscore = self.dataset_dict[dataset][video_index]['gtscore'][...]


    return {'features':features,'gtscore':gtscore,'data_point': data_point}

In [5]:
train_dataset = MultiH5Loader(split_file,split_name_train,cross_val_idx,feature_name,included_dataset)
val_dataset = MultiH5Loader(split_file,split_name_train,cross_val_idx,feature_name,included_dataset)

This dataloader basically returns the following

In [6]:
batch_dict = next(iter(train_dataset))

In [7]:
batch_dict['features'].shape

(935, 1024)

I've also implemented a dataloader which pads the input to all be the same length

In [11]:
train_dataloader =  DataLoader(train_dataset,shuffle=True,collate_fn=batch_collate_fn,batch_size = 5)
val_dataloader =  DataLoader(val_dataset,shuffle=False)

In [17]:
batch_padded = next(iter(train_dataloader))

In [18]:
batch_padded['mask'].shape

torch.Size([5, 1294])

In [7]:
class MetadataStore:
    def __init__(self, datasets:list|dict):
        # The dataset dict paths should also allow you to override and add custom h5's incase the h5's deviate (different shot boundaries,fps sampling etc)

        self.datasets = datasets
        self.files = {}

    def open(self):
        if isinstance(self.datasets, dict):
            self.files = {dataset:h5py.File(paths) for dataset,paths in self.datasets.items()}
        elif isinstance(self.datasets, list):
            self.files = {
                dataset: h5py.File(
                    f"Data/Metadata/{dataset}_metadata.h5", "r"
                )
                for dataset in self.datasets
            }
        else:
            raise TypeError("provide a list of included datasets, or a set of file paths")

    def get(self, dataset, video_key):
        f = self.files[dataset]
        group = f[video_key]

        metadata = {
            "positions": group["positions"][...],
            "n_frames": int(group["n_frames"][...]),
            "shot_bounds": group["shot_bounds"][...]
        }


        return metadata

    def get_gt(self,dataset,video_key):
        f = self.files[dataset]
        group = f[video_key]

        metadata = {
            "user_score": group["user_score"][...],
            "user_summary": group["user_summary"][...]
        }


        return metadata


    def close(self):
        for f in self.files.values():
            f.close()
        self.files.clear()

# Some general notes I will delete later
The post processing dict needs to ahve 

In [8]:
# Training of basic models which only require a single loss function to optimize
#TODO: Update metadata stores to also return the GT features we want to use
#TODO
class BaseTrainer(pl.LightningModule):

    def __init__(self,model:nn.Module,datasets:list|dict,eval_type:dict[str],post_process_dict:dict[str],criterion:Callable = F.mse_loss,eval_criterion='corr'):
        super().__init__()
        self.model = model
        self.eval_type = eval_type # Stores how each dataset has to be evaluated 
        self.metadata_stores = MetadataStore(datasets)
        self.metadata_stores.open()
        self.eval_criterion = eval_criterion
        self.post_process_dict = post_process_dict
        #TODO, add validation keys to exclude for hyper-parameter logging
        self.criterion = criterion

    def training_step(self, batch, batch_idx):
        x, y= batch['features'],batch['gtscore']
        mask = None
        if "mask" in batch.keys():
            mask = batch['mask']
        # Forward pass
        y_pred = self.model(x)
        
        # Calculate loss (MSE)
        loss = self.criterion(y_pred, y, mask) if mask is not None else self.criterion(y_pred, y)
        
        # Log the loss
        self.log('train_loss', loss)
        return loss

    def validation_step(self,batch,batch_idx):
        x,y,video_key = batch['features'],batch['gtscore'],batch['data_point'] # This might need to be changed to a dict, check with collate function
        y_pred = self.model(x) #TODO: maybe change this to a dictionary output.
        y = y.to('cpu')
        dataset,video_index = video_key[0].split('/')
        metadata = self.metadata_stores.get(dataset,video_index)
        ground_truth_data = self.metadata_stores.get_gt(dataset,video_index)
        eval_type = self.eval_type[dataset]
        post_process = self.post_process_dict[dataset]
        if self.eval_criterion =='corr':
            result_dict = process_and_route_single(y_pred,y,metadata,ground_truth_data,eval_type,post_process,self.eval_criterion)
            self.log('kendall',result_dict['kendall'],prog_bar = True,on_epoch=True)
            self.log('spearman',result_dict['spearman'],prog_bar = True,on_epoch=True)
        elif self.eval_criterion == 'f1':
            f1 = process_and_route_single(y_pred,y,metadata,ground_truth_data,eval_type,post_process,self.eval_criterion)
            self.log('f1',f1)

    def on_train_end(self):
        self.metadata_stores.close()

    # TODO: Also feed in training parameters to the model
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-5)
        return optimizer
        

In [9]:
model = PGL_SUM()

In [10]:
import torch

In [11]:
eval_type_dict = {'tvsum':'user_score'}
post_process_dict = {'tvsum':'upsample'}

In [12]:
def mse_squish(pred:torch.tensor,gt:torch.tensor):
    return F.mse_loss(pred.squeeze(),gt.squeeze())

In [13]:
lightning_module = BaseTrainer(model,datasets=included_dataset,eval_type=eval_type_dict,post_process_dict=post_process_dict)

In [14]:
trainer = pl.Trainer(max_epochs = 10)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [15]:
trainer.fit(lightning_module,train_dataloader,val_dataloader)

You are using a CUDA device ('NVIDIA GeForce RTX 4060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name  | Type    | Params | Mode 
------------------------------------------
0 | model | PGL_SUM | 5.2 M  | train
------------------------------------------
5.2 M     Trainable params
0         Non-trainable params
5.2 M     Total params
20.996    Total estimated model params size (MB)
19        Modules in train mode
0         Modules in eval mode


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 1. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
/home/aash/miniconda3/envs/MemorabilityEnv/lib/python3.10/site-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (40) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 9: 100%|██████████| 40/40 [00:00<00:00, 40.83it/s, v_num=10, kendall=0.213, spearman=0.278]   

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 40/40 [00:01<00:00, 37.38it/s, v_num=10, kendall=0.213, spearman=0.278]


In [14]:
out = model(torch.randn(1,332,1024))

In [16]:
out[0].shape

torch.Size([1, 332])